# Inverse PDE Full Run in Google Colab

This notebook sets up the project, trains, evaluates, and exports results for the inverse PDE pipeline.

Recommended runtime: GPU (T4, L4, or better).

## 1) Runtime Setup
In Colab: Runtime -> Change runtime type -> GPU

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
PROJECT_DIR = Path('/content/inverse_pde')

if not PROJECT_DIR.exists():
    if '<your-user>' in REPO_URL:
        raise ValueError('Set REPO_URL to your actual repository URL before running this cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

## 2) Configure Runs
Set the run toggles below. The full-size runs are expensive and can take multiple hours.

In [ ]:
RUN_BASELINE = True
RUN_NONSMOOTH = True

BASELINE_CONFIG = 'configs/full_run_d96.yaml'
NONSMOOTH_CONFIG = 'configs/nonsmooth.yaml'

DATA_DIR = 'data/generated'
OUT_BASELINE = 'outputs_colab_full_run_d96'
OUT_NONSMOOTH = 'outputs_colab_nonsmooth'
RES_BASELINE = 'results_colab_full_run_d96'
RES_NONSMOOTH = 'results_colab_nonsmooth'

## 3) Train Models
Each train command will auto-generate dataset shards if missing.

In [ ]:
if RUN_BASELINE:
    !python main.py --mode train --config {BASELINE_CONFIG} --data-dir {DATA_DIR} --output-dir {OUT_BASELINE}

In [ ]:
if RUN_NONSMOOTH:
    !python main.py --mode train --config {NONSMOOTH_CONFIG} --data-dir {DATA_DIR} --output-dir {OUT_NONSMOOTH}

## 4) Resolve Best Checkpoints

In [ ]:
from pathlib import Path

def newest_ckpt(output_dir: str) -> str:
    ckpt_dir = Path(output_dir) / 'checkpoints'
    files = sorted(ckpt_dir.glob('*.pt'), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f'No checkpoints found in {ckpt_dir}')
    return str(files[-1])

ckpt_baseline = newest_ckpt(OUT_BASELINE) if RUN_BASELINE else None
ckpt_nonsmooth = newest_ckpt(OUT_NONSMOOTH) if RUN_NONSMOOTH else None

print('Baseline checkpoint:', ckpt_baseline)
print('Nonsmooth checkpoint:', ckpt_nonsmooth)

## 5) Evaluate
Evaluation includes baselines, OOD checks, calibration metrics, and figures.

In [ ]:
if RUN_BASELINE and ckpt_baseline:
    !python main.py --mode evaluate --config {BASELINE_CONFIG} --checkpoint {ckpt_baseline} --data-dir {DATA_DIR} --results-dir {RES_BASELINE}

In [ ]:
if RUN_NONSMOOTH and ckpt_nonsmooth:
    !python main.py --mode evaluate --config {NONSMOOTH_CONFIG} --checkpoint {ckpt_nonsmooth} --data-dir {DATA_DIR} --results-dir {RES_NONSMOOTH}

## 6) Build a Comparison Table
This collects selected metrics from each run into a single CSV.

In [ ]:
import json
import pandas as pd
from pathlib import Path

rows = []
for name, res_dir in [('full_run_d96', RES_BASELINE), ('nonsmooth', RES_NONSMOOTH)]:
    p = Path(res_dir) / 'metrics.json'
    if not p.exists():
        continue
    with open(p, 'r', encoding='utf-8') as f:
        m = json.load(f)
    rows.append({
        'run': name,
        'rmse_mean': m.get('rmse_mean'),
        'ece': m.get('ece'),
        'coverage': m.get('coverage'),
        'pinn_rmse_mean': (m.get('baselines') or {}).get('pinn', {}).get('rmse_mean'),
        'pinn_convergence_fraction': (m.get('baselines') or {}).get('pinn', {}).get('convergence_fraction'),
    })

summary_dir = Path('results_final_summary')
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / 'comparison_colab.csv'
df = pd.DataFrame(rows)
df.to_csv(summary_path, index=False)
display(df)
print('Saved:', summary_path)

## 7) Download Artifacts
Use this to download trained outputs and result folders.

In [ ]:
import shutil
from pathlib import Path

zip_path = Path('/content/inverse_pde_colab_results.zip')
if zip_path.exists():
    zip_path.unlink()

targets = [
    OUT_BASELINE, OUT_NONSMOOTH,
    RES_BASELINE, RES_NONSMOOTH,
    'results_final_summary'
]
existing = [t for t in targets if Path(t).exists()]

tmp_root = Path('/content/_inverse_pde_export')
if tmp_root.exists():
    shutil.rmtree(tmp_root)
tmp_root.mkdir(parents=True, exist_ok=True)

for t in existing:
    src = Path(t)
    dst = tmp_root / src.name
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)

shutil.make_archive('/content/inverse_pde_colab_results', 'zip', tmp_root)
print('Created zip:', zip_path)

from google.colab import files
files.download(str(zip_path))